In [ ]:
import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.7)

mp_draw = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)  # webcam mặc định

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(frame_rgb)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            # Lấy tọa độ 5 đầu ngón tay: [4, 8, 12, 16, 20]
            tips = [4, 8, 12, 16, 20]
            for tip_id in tips:
                x = int(hand_landmarks.landmark[tip_id].x * frame.shape[1])
                y = int(hand_landmarks.landmark[tip_id].y * frame.shape[0])
                cv2.circle(frame, (x, y), 10, (0, 255, 0), -1)
            
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

    cv2.imshow("Finger Tracking", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


d:\Github\Machine-Learning-Studies\tf-env\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [3]:
import cv2
import mediapipe as mp
import numpy as np

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)
cap = cv2.VideoCapture(0)

prev_positions = None

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            positions = []
            h, w, _ = frame.shape
            
            # Lấy toàn bộ 21 landmark
            for idx, lm in enumerate(hand_landmarks.landmark):
                x, y = int(lm.x * w), int(lm.y * h)
                positions.append((x, y))
                # Vẽ chấm đỏ tại từng landmark
                cv2.circle(frame, (x, y), 5, (0, 0, 255), -1)
                cv2.putText(frame, str(idx), (x+5, y-5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

            # Tính vector di chuyển
            if prev_positions is not None:
                movement = np.array(positions) - np.array(prev_positions)
                print("Movement:", movement)

            prev_positions = positions

    cv2.imshow("Hand Tracking", frame)
    if cv2.waitKey(1) == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()


Movement: [[  7 -17]
 [ -2 -14]
 [-12 -24]
 [-14 -30]
 [-17 -22]
 [ -9 -30]
 [ -6 -26]
 [ -3 -24]
 [ -4 -21]
 [-13 -26]
 [-15 -26]
 [-16 -21]
 [-19 -19]
 [-11 -27]
 [ -2 -22]
 [  2 -15]
 [  5 -12]
 [ -5 -26]
 [  1 -28]
 [  1 -23]
 [  1 -22]]
Movement: [[-18 -14]
 [-13 -21]
 [ -8 -24]
 [ -6 -25]
 [ -5 -31]
 [ -7 -28]
 [ -8 -29]
 [ -8 -29]
 [ -9 -28]
 [ -5 -27]
 [ -6 -26]
 [ -6 -27]
 [ -7 -25]
 [ -4 -24]
 [  6 -28]
 [ 11 -30]
 [ 10 -31]
 [ -4 -22]
 [  7 -21]
 [ 12 -25]
 [ 13 -28]]
Movement: [[  0 -10]
 [ -8 -16]
 [-10 -22]
 [-11 -28]
 [-13 -24]
 [ -9 -20]
 [ -9 -20]
 [-10 -21]
 [ -9 -21]
 [ -9 -20]
 [ -8 -18]
 [ -7 -17]
 [ -5 -18]
 [ -7 -19]
 [ -1 -26]
 [  1 -28]
 [  1 -26]
 [ -3 -17]
 [  1 -27]
 [  5 -43]
 [  9 -56]]
Movement: [[ -11   -3]
 [  -6   -6]
 [  -6  -11]
 [  -4  -11]
 [  -3  -12]
 [  -5  -12]
 [  -4  -10]
 [  -3   -9]
 [  -3   -9]
 [  -5   -9]
 [  -5  -10]
 [  -4   -9]
 [  -4  -10]
 [  -2   -8]
 [  -3  -23]
 [   2  -67]
 [   6 -107]
 [  -2   -8]
 [  -2  -18]
 [   1  -38]
 [  

In [63]:
import cv2
import mediapipe as mp
import numpy as np
import time

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)
cap = cv2.VideoCapture(0)

recording = False
record_data = []
start_time = None

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    positions = None
    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            h, w, _ = frame.shape
            positions = []
            for idx, lm in enumerate(hand_landmarks.landmark):  # Lấy 21 điểm
                positions.append((lm.x, lm.y))  # normalized
                
                # Vẽ điểm
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)

    # Nếu đang ghi dữ liệu
    if recording and positions is not None:
        record_data.append(positions)

        # Sau 1 giây
        if time.time() - start_time >= 0.5:
            filename = f"hand_record_{int(time.time())}.npy"
            np.save(filename, np.array(record_data))
            print(f"✅ Saved: {filename}, shape={np.array(record_data).shape}")
            record_data = []
            recording = False

    cv2.putText(frame, "Press '1' to record 1s hand motion", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    cv2.imshow("Hand Tracking", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == 27:  # ESC thoát
        break
    elif key == ord('1') and not recording:
        print("🎥 Start recording...")
        recording = True
        start_time = time.time()
        record_data = []

cap.release()
cv2.destroyAllWindows()


🎥 Start recording...
✅ Saved: hand_record_1755577434.npy, shape=(15, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1755577436.npy, shape=(16, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1755577437.npy, shape=(15, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1755577438.npy, shape=(16, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1755577439.npy, shape=(15, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1755577441.npy, shape=(16, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1755577443.npy, shape=(16, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1755577444.npy, shape=(16, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1755577445.npy, shape=(17, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1755577446.npy, shape=(15, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1755577447.npy, shape=(17, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1755577448.npy, shape=(15, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1755577449.npy, shape=(16, 21, 2)
🎥 Start reco

In [68]:
import os

folder_path = r"D:\Github\Machine-Learning-Studies\Handtracking\command\nocommand"  # đổi thành folder của bạn
files = [f for f in os.listdir(folder_path) if f.endswith(".npy")]

# sắp xếp file nếu muốn theo thứ tự alphabet
files.sort()

for idx, filename in enumerate(files, 1):  # bắt đầu STT từ 1
    old_path = os.path.join(folder_path, filename)
    new_name = f"nocommand{idx}.npy"
    new_path = os.path.join(folder_path, new_name)
    os.rename(old_path, new_path)

print(f"✅ Đã đổi tên {len(files)} file thành swipeleft{{x}}.npy")

✅ Đã đổi tên 475 file thành swipeleft{x}.npy


In [69]:
import os
import numpy as np

def load_all_sequences(base_folder):
    class_names = sorted(os.listdir(base_folder))  # lấy tên các class từ tên thư mục
    sequences = []
    labels = []
    
    for idx, class_name in enumerate(class_names):
        class_folder = os.path.join(base_folder, class_name)
        if not os.path.isdir(class_folder):
            continue
        
        files = [f for f in os.listdir(class_folder) if f.endswith(".npy")]
        
        for f in files:
            seq_path = os.path.join(class_folder, f)
            seq = np.load(seq_path)  # shape: (frames, features)
            sequences.append(seq)
            labels.append(idx)  # gán nhãn bằng index của class
    
    return np.array(sequences, dtype=object), np.array(labels), class_names


# Ví dụ dùng
folder_path = r"D:\Github\Machine-Learning-Studies\Handtracking\command"
X, y, class_names = load_all_sequences(folder_path)

print("Classes:", class_names)
print("Số mẫu:", len(X))

for i in range (len(X)):
    print("Shape :", X[i].shape)
    print("Label :", y[i])


Classes: ['nocommand', 'swipeleft', 'swiperight']
Số mẫu: 922
Shape : (15, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (15, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (17, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (17, 21, 2)
Label : 0
Shape : (15, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (15, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (15, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (15, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : (16, 21, 2)
Label : 0
Shape : 

In [70]:
import numpy as np
from scipy.interpolate import interp1d

def resize_frames(seq, target_len=16):
    """
    Resize 1 sequence (frames, joints, coords) thành target_len frame.
    seq shape: (T, J, C)
    """
    old_len = seq.shape[0]
    # vị trí index ban đầu và mới (chuẩn hóa về [0,1])
    x_old = np.linspace(0, 1, old_len)
    x_new = np.linspace(0, 1, target_len)

    f = interp1d(x_old, seq, axis=0)  # nội suy theo trục time
    return f(x_new)

# Ví dụ áp dụng cho cả dataset
X_resized = np.array([resize_frames(seq, 16) for seq in X])
print("X_resized shape:", X_resized.shape)  # (num_samples, 16, 21, 2)


X_resized shape: (922, 16, 21, 2)


In [71]:
for i, s in enumerate(X_resized):
    print(f"Seq {i}: shape={s.shape}")

Seq 0: shape=(16, 21, 2)
Seq 1: shape=(16, 21, 2)
Seq 2: shape=(16, 21, 2)
Seq 3: shape=(16, 21, 2)
Seq 4: shape=(16, 21, 2)
Seq 5: shape=(16, 21, 2)
Seq 6: shape=(16, 21, 2)
Seq 7: shape=(16, 21, 2)
Seq 8: shape=(16, 21, 2)
Seq 9: shape=(16, 21, 2)
Seq 10: shape=(16, 21, 2)
Seq 11: shape=(16, 21, 2)
Seq 12: shape=(16, 21, 2)
Seq 13: shape=(16, 21, 2)
Seq 14: shape=(16, 21, 2)
Seq 15: shape=(16, 21, 2)
Seq 16: shape=(16, 21, 2)
Seq 17: shape=(16, 21, 2)
Seq 18: shape=(16, 21, 2)
Seq 19: shape=(16, 21, 2)
Seq 20: shape=(16, 21, 2)
Seq 21: shape=(16, 21, 2)
Seq 22: shape=(16, 21, 2)
Seq 23: shape=(16, 21, 2)
Seq 24: shape=(16, 21, 2)
Seq 25: shape=(16, 21, 2)
Seq 26: shape=(16, 21, 2)
Seq 27: shape=(16, 21, 2)
Seq 28: shape=(16, 21, 2)
Seq 29: shape=(16, 21, 2)
Seq 30: shape=(16, 21, 2)
Seq 31: shape=(16, 21, 2)
Seq 32: shape=(16, 21, 2)
Seq 33: shape=(16, 21, 2)
Seq 34: shape=(16, 21, 2)
Seq 35: shape=(16, 21, 2)
Seq 36: shape=(16, 21, 2)
Seq 37: shape=(16, 21, 2)
Seq 38: shape=(16, 21,

In [72]:
import numpy as np
from tensorflow.keras.utils import to_categorical

X_gru = X_resized.reshape(X_resized.shape[0], 16, -1)  # (samples, timesteps, features=42)

# 3. One-hot label
y_categorical = to_categorical(y, num_classes=3)  # shape: (samples, 2)

print("X_gru shape:", X_gru.shape)
print("y_categorical shape:", y_categorical.shape)


X_gru shape: (922, 16, 42)
y_categorical shape: (922, 3)


In [73]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, GRU

# Chia train/test
X_train, X_test, y_train, y_test = train_test_split(X_gru, y_categorical, test_size=0.2, random_state=42)

# Model đơn giản dùng GRU
model = Sequential([
    GRU(64, return_sequences=False, input_shape=(X_gru.shape[1], X_gru.shape[2])),
    Dense(32, activation='relu'),
    Dense(y_categorical.shape[1], activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=30, batch_size=8)


Epoch 1/30
93/93 [==============================] - 2s 8ms/step - loss: 0.8351 - accuracy: 0.6228 - val_loss: 0.4902 - val_accuracy: 0.8486
Epoch 2/30
93/93 [==============================] - 0s 5ms/step - loss: 0.3425 - accuracy: 0.8643 - val_loss: 0.2932 - val_accuracy: 0.8865
Epoch 3/30
93/93 [==============================] - 0s 5ms/step - loss: 0.2321 - accuracy: 0.9104 - val_loss: 0.2783 - val_accuracy: 0.8811
Epoch 4/30
93/93 [==============================] - 0s 5ms/step - loss: 0.1926 - accuracy: 0.9335 - val_loss: 0.2977 - val_accuracy: 0.9135
Epoch 5/30
93/93 [==============================] - 0s 5ms/step - loss: 0.1599 - accuracy: 0.9322 - val_loss: 0.2363 - val_accuracy: 0.9243
Epoch 6/30
93/93 [==============================] - 0s 5ms/step - loss: 0.1200 - accuracy: 0.9607 - val_loss: 0.2203 - val_accuracy: 0.9405
Epoch 7/30
93/93 [==============================] - 0s 5ms/step - loss: 0.1015 - accuracy: 0.9593 - val_loss: 0.1590 - val_accuracy: 0.9405
Epoch 8/30
93/93 [==

In [75]:
model.save("action_gru_model.h5")

In [4]:
import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.models import load_model
from collections import deque
import time
import os

# ========= CONFIG (bạn có thể sửa) =========
MODEL_PATH = "action_gru_model.h5"
# Nếu bạn biết đúng thứ tự label lúc train thì đặt ở đây:
ACTIONS = ["nocommand", "swipeleft", "swiperight"]

# ========= LOAD MODEL & AUTOCONFIG =========
model = load_model(MODEL_PATH)

# Input shape: (None, T, F)
_, TIMESTEPS, FEATURES = model.input_shape
if TIMESTEPS is None or FEATURES is None:
    # fallback an toàn
    TIMESTEPS = 32
    FEATURES = 42  # mặc định 21 điểm * (x,y)

# Output classes
NUM_CLASSES = model.output_shape[-1]

# Chuẩn hóa/khớp ACTIONS với số lớp
if ACTIONS is None or len(ACTIONS) != NUM_CLASSES:
    # Nếu chỉ có 1 nhãn "swiperight" mà model có 2 class → tạo generic
    print(f"[WARN] ACTIONS len={0 if ACTIONS is None else len(ACTIONS)} "
          f"không khớp NUM_CLASSES={NUM_CLASSES}. "
          f"Auto-generate nhãn: class_0..class_{NUM_CLASSES-1}")
    ACTIONS = [f"class_{i}" for i in range(NUM_CLASSES)]

# Suy ra cách trích feature theo FEATURES
# 21 điểm: (x,y)=42, (x,y,z)=63
if FEATURES in (42, 63):
    USE_21_POINTS = True
    USE_Z = (FEATURES == 63)
    POINTS_PER_FRAME = 21
# 5 fingertips (x,y)=10, (x,y,z)=15
elif FEATURES in (10, 15):
    USE_21_POINTS = False
    USE_Z = (FEATURES == 15)
    POINTS_PER_FRAME = 5
else:
    raise ValueError(f"FEATURES={FEATURES} không khớp các cấu hình quen thuộc "
                     f"(10/15/42/63). Kiểm tra lại model & extractor.")

EXPECTED_FEATURES = POINTS_PER_FRAME * (3 if USE_Z else 2)
if EXPECTED_FEATURES != FEATURES:
    raise ValueError(
        f"Model expects {FEATURES} features/frame, "
        f"nhưng extractor dự kiến {EXPECTED_FEATURES}."
    )

# ========= MEDIAPIPE SETUP =========
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

# ========= BUFFER =========
seq_buffer = deque(maxlen=TIMESTEPS)

# ========= CAMERA =========
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Không mở được camera (index 0).")

# Bạn có thể chỉnh độ phân giải nếu muốn:
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

last_pred = None
last_prob = 0.0
t0 = time.time()
frame_count = 0

# Nếu dùng 5 fingertips
TIPS_ID = [4, 8, 12, 16, 20] if not USE_21_POINTS else None

def extract_features(results):
    """Trả về vector features (FEATURES,) phù hợp với model."""
    if results.multi_hand_landmarks:
        hand = results.multi_hand_landmarks[0]
        feats = []
        idxs = range(21) if USE_21_POINTS else TIPS_ID
        for i in idxs:
            lm = hand.landmark[i]
            if USE_Z:
                feats.extend([lm.x, lm.y, lm.z])
            else:
                feats.extend([lm.x, lm.y])
        return np.asarray(feats, dtype=np.float32)
    # Không thấy tay → zeros
    return np.zeros((FEATURES,), dtype=np.float32)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    # Vẽ landmarks (để quan sát)
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

    # Trích features và đẩy vào buffer
    feat = extract_features(results)
    seq_buffer.append(feat)

    # Khi đủ TIMESTEPS thì dự đoán
    if len(seq_buffer) == TIMESTEPS:
        x = np.expand_dims(np.asarray(seq_buffer, dtype=np.float32), axis=0)  # (1, T, F)
        pred = model.predict(x, verbose=0)[0]  # (NUM_CLASSES,)
        pred_idx = int(np.argmax(pred))
        pred_prob = float(np.max(pred))

        # Bảo vệ index (dù đã đồng bộ ACTIONS ở trên)
        if 0 <= pred_idx < len(ACTIONS):
            last_pred = ACTIONS[pred_idx]
            last_prob = pred_prob
        else:
            last_pred = f"idx_{pred_idx}"
            last_prob = pred_prob

    # Hiển thị kết quả
    if last_pred is not None:
        cv2.putText(frame, f"{last_pred} ({last_prob:.2f})", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (50, 220, 50), 2)

    # Tính FPS
    frame_count += 1
    now = time.time()
    if now - t0 >= 1.0:
        fps = frame_count / (now - t0)
        t0 = now
        frame_count = 0
        cv2.putText(frame, f"FPS: {fps:.1f}", (10, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (200, 200, 200), 2)
    else:
        cv2.putText(frame, "FPS: ...", (10, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (200, 200, 200), 2)

    # Overlay thông tin model (1 lần/khung cho tiện debug)
    cv2.putText(frame, f"T={TIMESTEPS}, F={FEATURES}, classes={NUM_CLASSES}", (10, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (180, 180, 180), 2)

    cv2.imshow("Action Recognition (GRU + MediaPipe)", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
hands.close()
cv2.destroyAllWindows()